# Conditional TabM: all + Futures/Regular 60:40
전체 모델과 경기 유형별 모델을 각각 학습하고, 2024년으로 마지막 MLP block + output layer fine-tuning한 뒤 제출 ZIP을 생성합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
PROJECT_DIR = Path('/content/drive/MyDrive/aimers')
TRAIN_DIR = PROJECT_DIR / 'train_chan_3'
DATA_DIR = PROJECT_DIR / 'open' / 'data'
ARTIFACTS = TRAIN_DIR / 'artifacts_conditional'
for p in [TRAIN_DIR / 'train_conditional.py', DATA_DIR / 'train.csv', TRAIN_DIR / 'selected_config.json']:
    assert p.exists(), f'경로를 확인하세요: {p}'
print('project:', PROJECT_DIR)
print('train:', DATA_DIR / 'train.csv')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(TRAIN_DIR / 'requirements_colab.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Colab 런타임을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
cmd = [
    sys.executable, '-u', str(TRAIN_DIR / 'train_conditional.py'),
    '--data-dir', str(DATA_DIR),
    '--config', str(TRAIN_DIR / 'selected_config.json'),
    '--output-dir', str(ARTIFACTS),
    '--device', 'cuda',
    '--stage2-epochs', '1',
    '--fine-tune-scope', 'last_block',
]
result = subprocess.run(cmd, check=False)
assert result.returncode == 0, f'conditional 학습 실패(returncode={result.returncode})'

In [ ]:
BUILD = TRAIN_DIR / 'build_submission.py'
BUNDLE = PROJECT_DIR / 'submit_tabm_conditional'
ZIP = PROJECT_DIR / 'submit_tabm_conditional.zip'
cmd = [
    sys.executable, '-u', str(BUILD),
    '--artifacts-dir', str(ARTIFACTS),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(BUNDLE),
    '--zip', str(ZIP),
    '--device', 'cuda',
]
result = subprocess.run(cmd, check=False)
assert result.returncode == 0, f'제출 ZIP 생성 실패(returncode={result.returncode})'
print('제출 파일:', ZIP)